# Analyze experiment results

Pulls every result + artifact for a given `experiment_id` from the Telemetry Service (`http://localhost:8004`), prints raw QoE / transport / contextual context, downloads the pcap, and plots a throughput time series.

Defaults to **`shell-40.0mbps-100.0ms-cubic-001`** (the most recent NDT speedtest experiment). Override with the `EXPERIMENT_ID` env var if needed.

Requires: `requests`, `scapy`, `matplotlib`.

In [ ]:
import os, sys
from pathlib import Path

# Make `services/analysis` importable when this notebook is opened from the
# repo root or from the services/ folder.
_here = Path.cwd()
for cand in [_here, _here.parent, _here.parent.parent]:
    if (cand / 'services' / 'analysis' / '__init__.py').exists():
        sys.path.insert(0, str(cand))
        break
    if (cand / 'analysis' / '__init__.py').exists():
        sys.path.insert(0, str(cand))
        break

try:
    from services.analysis import (
        TelemetryClient, print_result_summary, print_qoe_metrics,
        print_transport_state, extract_speedtest, extract_iperf, extract_ping, extract_wget,
        load_pcap, summarize_pcap,
        plot_throughput, plot_icmp_rtt, plot_packet_size,
    )
except ModuleNotFoundError:
    from analysis import (
        TelemetryClient, print_result_summary, print_qoe_metrics,
        print_transport_state, extract_speedtest, extract_iperf, extract_ping, extract_wget,
        load_pcap, summarize_pcap,
        plot_throughput, plot_icmp_rtt, plot_packet_size,
    )

import matplotlib.pyplot as plt
%matplotlib inline

EXPERIMENT_ID = os.environ.get('EXPERIMENT_ID', 'shell-40.0mbps-100.0ms-cubic-001')
TELEMETRY_BASE = os.environ.get('TELEMETRY_BASE', 'http://localhost:8004')
PCAP_DIR = Path('./_pcap_cache').resolve()

print(f'experiment_id  : {EXPERIMENT_ID}')
print(f'telemetry_base : {TELEMETRY_BASE}')
print(f'pcap cache dir : {PCAP_DIR}')

## 1. Health check + fetch all results for this experiment

In [ ]:
client = TelemetryClient(TELEMETRY_BASE)
print('telemetry health:', client.health())

bundle = client.fetch_experiment(EXPERIMENT_ID)
print(f'\nfound {len(bundle.results)} result(s) for {EXPERIMENT_ID}')
for r in bundle.results:
    print(f'  - result_id={r.get("result_id")}  trial={r.get("trial_number")}  status={r.get("status")}')
    arts = bundle.artifacts_by_result.get(r['result_id'], [])
    for a in arts:
        print(f'      artifact: {a.get("artifact_type"):8s}  {a.get("filename")}  ({a.get("size_bytes")} bytes)')

## 2. Result summary + four-layer contextual tree

In [ ]:
result = bundle.latest
if result is None:
    raise RuntimeError(f'no telemetry rows for {EXPERIMENT_ID}')

# get_result returns the full row (some list endpoints trim fields)
result = client.get_result(result['result_id'])
print_result_summary(result)

## 3. Raw QoE metrics + transport state

The shape of `qoe_metrics` depends on the workflow (speedtest vs iperf vs ping vs youtube). The first cell dumps it verbatim; the second tries the three workflow-specific extractors and prints whichever one matched.

In [ ]:
print_qoe_metrics(result)
print()
print_transport_state(result)

In [ ]:
for label, fn in [('speedtest', extract_speedtest), ('iperf', extract_iperf), ('ping', extract_ping), ('wget', extract_wget)]:
    extracted = fn(result)
    if extracted:
        print(f'[{label}]')
        for k, v in extracted.items():
            print(f'  {k}: {v}')

## 4. Download + summarize the pcap

In [ ]:
pcap_paths = client.download_pcaps(bundle, PCAP_DIR, result_id=result['result_id'])
if not pcap_paths:
    raise RuntimeError(f'no pcap artifact for result {result["result_id"]}')
pcap_path = pcap_paths[0]
print('pcap:', pcap_path)

pkts = load_pcap(pcap_path)
print('summary:', summarize_pcap(pkts))

## 5. Plot throughput + per-packet size + (if any) ICMP RTT

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10))
plot_throughput(pkts, bin_seconds=0.1, unit='mbps', ax=axes[0],
                title=f'Throughput — {EXPERIMENT_ID}')
plot_packet_size(pkts, ax=axes[1])
plot_icmp_rtt(pkts, ax=axes[2])
plt.tight_layout()
plt.show()

## 6. (Optional) repeat for every result/trial in the bundle

In [ ]:
for res in bundle.results:
    rid = res['result_id']
    print('=' * 80)
    print(f'result_id={rid}  trial={res.get("trial_number")}  status={res.get("status")}')
    full = client.get_result(rid)
    extracted = extract_speedtest(full) or extract_iperf(full) or extract_ping(full)
    if extracted:
        for k, v in extracted.items():
            print(f'  {k}: {v}')
    paths = client.download_pcaps(bundle, PCAP_DIR, result_id=rid)
    if not paths:
        print('  (no pcap)')
        continue
    pkts_i = load_pcap(paths[0])
    print('  pcap summary:', summarize_pcap(pkts_i))